# Loading the pretrained model

In [ ]:
from pathlib import Path
import torch

CHECKPOINT_PATH = Path(
    "/content/model_fineweb_edu_pilot.pt"
)

TOKENIZER_PATH = Path(
    "/content/tokenizer_qwen40k.json"
)

assert CHECKPOINT_PATH.is_file(), (
    f"Model checkpoint not found: {CHECKPOINT_PATH}"
)

assert TOKENIZER_PATH.is_file(), (
    f"Tokenizer file not found: {TOKENIZER_PATH}"
)

print("Checkpoint and tokenizer files found.")

In [ ]:
import torch

# Use the same architecture configuration as the pilot run.
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

# Reconstruct the model architecture.
model = Qwen3Model(QWEN3_CONFIG)

# Load the checkpoint created by your pilot training.
# Use weights_only=False only for a checkpoint you trust.
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

# Restore the learned model parameters.
model.load_state_dict(
    checkpoint["model"],
    strict=True,
)

model = model.to(device)
model.eval()

print("Model loaded successfully.")
print("Device:", device)
print("Architecture:", model.model_name)
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Checkpoint config:", checkpoint["config"])

In [ ]:
tokenizer = Qwen3Tokenizer(
    str(TOKENIZER_PATH)
)

print("Tokenizer loaded successfully.")
print("Vocabulary size:", tokenizer.vocab_size)
print("EOS token ID:", tokenizer.eos_token_id)
print("PAD token ID:", tokenizer.pad_token_id)

assert tokenizer.vocab_size == QWEN3_CONFIG["vocab_size"], (
    f"Tokenizer vocabulary ({tokenizer.vocab_size}) does not match "
    f"model vocabulary ({QWEN3_CONFIG['vocab_size']})."
)

In [ ]:
# Test the tokenizer.
test_text = "How can I improve rice crop growth?"

token_ids = tokenizer.encode(test_text)

assert token_ids, "Tokenizer returned no token IDs."
assert min(token_ids) >= 0
assert max(token_ids) < QWEN3_CONFIG["vocab_size"]

# Test a forward pass without updating the weights.
input_ids = torch.tensor(
    [token_ids],
    dtype=torch.long,
    device=device,
)

with torch.no_grad():
    logits = model(input_ids)

print("Input shape:", tuple(input_ids.shape))
print("Logits shape:", tuple(logits.shape))
print("Output vocabulary:", logits.shape[-1])

assert logits.shape[-1] == QWEN3_CONFIG["vocab_size"]
assert torch.isfinite(logits).all()

print("Step 1 verification passed.")

# Preparing the dataset

In [1]:
!pip install -q datasets pandas



In [ ]:
from datasets import load_dataset

DATASET_ID = "vicharanashala/ajrasakha-dataset-v1"

dataset = load_dataset(DATASET_ID)

print("Available splits:", list(dataset.keys()))

for split_name, split_data in dataset.items():
    print(f"\n{split_name}: {len(split_data):,} rows")
    print("Columns:", split_data.column_names)

In [ ]:
import pandas as pd

train_df = dataset["train"].to_pandas()
validation_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

print("Training examples:")
display(train_df[["question", "final_answer"]].head(5))

print("\nMissing values in training data:")
print(
    train_df[["question", "final_answer"]]
    .isna()
    .sum()
)

print("\nTraining data shape:", train_df.shape)

In [2]:
import re

INVALID_ANSWERS = {
    "n/a",
    "na",
    "none",
    "null",
    "no answer",
    "not available",
    "test",
}

def clean_text(value):
    if not isinstance(value, str):
        return ""
    return re.sub(r"\s+", " ", value).strip()

def clean_split(df):
    df = df.copy()

    # Normalize whitespace without changing the meaning of answers.
    df["question"] = df["question"].map(clean_text)
    df["final_answer"] = df["final_answer"].map(clean_text)

    # Remove missing or blank questions and answers.
    valid = (
        df["question"].ne("")
        & df["final_answer"].ne("")
    )

    # Remove obvious placeholder answers.
    invalid_answer = (
        df["final_answer"]
        .str.casefold()
        .isin(INVALID_ANSWERS)
    )

    df = df.loc[valid & ~invalid_answer].copy()

    # Remove exact duplicate question-answer pairs.
    df = df.drop_duplicates(
        subset=["question", "final_answer"]
    )

    return df.reset_index(drop=True)

train_clean = clean_split(train_df)
validation_clean = clean_split(validation_df)
test_clean = clean_split(test_df)

print("Rows before and after cleaning:")
print(f"Train:      {len(train_df):,} -> {len(train_clean):,}")
print(f"Validation: {len(validation_df):,} -> {len(validation_clean):,}")
print(f"Test:       {len(test_df):,} -> {len(test_clean):,}")

NameError: name 'train_df' is not defined

In [ ]:
def answer_quality_report(df, split_name):
    print(f"\n--- {split_name.upper()} QUALITY REPORT ---")

    answer_lengths = df["final_answer"].str.len()

    print("Rows:", len(df))
    print("Empty questions:", (df["question"].str.strip() == "").sum())
    print("Empty answers:", (df["final_answer"].str.strip() == "").sum())
    print("Answers under 30 characters:", (answer_lengths < 30).sum())
    print("Questions under 10 characters:",
          (df["question"].str.len() < 10).sum())

    if "final_answer_approval_count" in df.columns:
        print("\nApproval-count distribution:")
        print(
            df["final_answer_approval_count"]
            .value_counts(dropna=False)
            .sort_index()
            .head(20)
        )

    print("\nShortest answers for manual review:")
    display(
        df.assign(answer_length=answer_lengths)
          .sort_values("answer_length")
          [["question", "final_answer", "answer_length"]]
          .head(15)
    )

answer_quality_report(train_clean, "train")
answer_quality_report(validation_clean, "validation")
answer_quality_report(test_clean, "test")

In [3]:
import json
from urllib.parse import urlparse

def parse_sources(value):
    if isinstance(value, list):
        return value, None

    if isinstance(value, dict):
        return [value], None

    if not isinstance(value, str) or not value.strip():
        return [], "missing"

    try:
        parsed = json.loads(value)
        if isinstance(parsed, dict):
            parsed = [parsed]
        if isinstance(parsed, list):
            return parsed, None
        return [], "not_a_list"
    except (json.JSONDecodeError, TypeError):
        return [], "invalid_json"

def valid_http_url(value):
    if not isinstance(value, str):
        return False
    try:
        parsed = urlparse(value.strip())
        return parsed.scheme in {"http", "https"} and bool(parsed.netloc)
    except Exception:
        return False

def citation_audit(df):
    df = df.copy()

    parsed_results = df["final_answer_sources"].map(parse_sources)

    df["citation_parse_issue"] = parsed_results.map(
        lambda result: result[1] or ""
    )
    df["parsed_sources"] = parsed_results.map(
        lambda result: result[0]
    )
    df["citation_count"] = df["parsed_sources"].map(len)

    def check_entries(sources):
        if not sources:
            return "no_citations"

        issues = []
        for i, source in enumerate(sources):
            if not isinstance(source, dict):
                issues.append(f"entry_{i}_not_object")
                continue

            if not valid_http_url(source.get("source")):
                issues.append(f"entry_{i}_invalid_url")

            if not clean_text(source.get("sourceName", "")):
                issues.append(f"entry_{i}_missing_title")

        return ";".join(issues) if issues else ""

    df["citation_issues"] = df["parsed_sources"].map(check_entries)

    return df

train_audit = citation_audit(train_clean)

print("Citation audit summary:")
print("Missing citation field:",
      (train_audit["citation_parse_issue"] == "missing").sum())
print("Invalid citation JSON:",
      (train_audit["citation_parse_issue"] == "invalid_json").sum())
print("No citations:",
      (train_audit["citation_count"] == 0).sum())
print("Rows with citation issues:",
      (train_audit["citation_issues"] != "").sum())

display(
    train_audit.loc[
        train_audit["citation_issues"] != "",
        ["question", "final_answer", "final_answer_sources",
         "citation_issues"]
    ].head(20)
)

NameError: name 'train_clean' is not defined

In [ ]:
def normalize_question(text):
    text = clean_text(text).casefold()
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def duplicate_report(df, split_name):
    df = df.copy()
    df["question_key"] = df["question"].map(normalize_question)

    repeated = df[
        df.duplicated("question_key", keep=False)
    ].sort_values("question_key")

    print(f"\n--- {split_name.upper()} DUPLICATE REPORT ---")
    print("Rows involved in repeated questions:", len(repeated))

    if not repeated.empty:
        display(
            repeated[
                ["question", "final_answer", "normalised_crop",
                 "state", "district"]
            ].head(30)
        )

    answer_counts = (
        df.groupby("question_key")["final_answer"]
          .nunique()
    )
    conflict_keys = answer_counts[answer_counts > 1].index

    conflicts = df[df["question_key"].isin(conflict_keys)]

    print("Question groups with differing answers:", len(conflict_keys))
    if not conflicts.empty:
        display(
            conflicts[
                ["question", "final_answer", "normalised_crop",
                 "state", "district"]
            ].head(30)
        )

    return repeated, conflicts

train_repeated, train_conflicts = duplicate_report(
    train_clean, "train"
)

In [ ]:
def split_question_keys(df):
    return set(df["question"].map(normalize_question))

train_keys = split_question_keys(train_clean)
validation_keys = split_question_keys(validation_clean)
test_keys = split_question_keys(test_clean)

print("Train/validation question overlap:", len(train_keys & validation_keys))
print("Train/test question overlap:", len(train_keys & test_keys))
print("Validation/test question overlap:", len(validation_keys & test_keys))

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("ajrasakha_prepared")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_clean.to_parquet(
    OUTPUT_DIR / "train_clean.parquet", index=False
)
validation_clean.to_parquet(
    OUTPUT_DIR / "validation_clean.parquet", index=False
)
test_clean.to_parquet(
    OUTPUT_DIR / "test_clean.parquet", index=False
)

# Keep records with citation issues for manual review.
citation_review = train_audit[
    (train_audit["citation_issues"] != "")
    | (train_audit["citation_parse_issue"] != "")
]

citation_review.to_csv(
    OUTPUT_DIR / "citation_review.csv", index=False
)

print("Saved prepared files to:", OUTPUT_DIR.resolve())
print("Train:", len(train_clean))
print("Validation:", len(validation_clean))
print("Test:", len(test_clean))
print("Citation records for review:", len(citation_review))

In [ ]:
import json

def make_training_example(row):
    context_parts = []

    for column, label in [
        ("normalised_crop", "Crop"),
        ("state", "State"),
        ("district", "District"),
        ("season", "Season"),
        ("domain", "Topic"),
    ]:
        if column in row.index:
            value = clean_text(row[column])
            if value:
                context_parts.append(f"{label}: {value}")

    context = "\n".join(context_parts)

    question = clean_text(row["question"])
    answer = clean_text(row["final_answer"])

    if context:
        text = (
            f"{context}\n"
            f"Question: {question}\n"
            f"Answer: {answer}"
        )
    else:
        text = f"Question: {question}\nAnswer: {answer}"

    return {
        "text": text,
        "question_id": clean_text(row.get("question_id", "")),
    }

def export_jsonl(df, path):
    with open(path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            example = make_training_example(row)
            f.write(json.dumps(example, ensure_ascii=False) + "\n")

export_jsonl(
    train_clean,
    OUTPUT_DIR / "train.jsonl"
)
export_jsonl(
    validation_clean,
    OUTPUT_DIR / "validation.jsonl"
)
export_jsonl(
    test_clean,
    OUTPUT_DIR / "test.jsonl"
)

print("Training examples exported.")

In [ ]:
with open(
    OUTPUT_DIR / "train.jsonl",
    "r",
    encoding="utf-8"
) as f:
    example = json.loads(f.readline())

print(example["text"])

# Formatting the dataset into instruction-response conversations

In [ ]:
import json
from pathlib import Path

DEFAULT_SYSTEM = (
    "You are an agricultural assistant. Provide accurate, clear, "
    "practical advice. Consider the crop, location, season, and "
    "conditions when relevant. Do not invent facts or citations. "
    "When the available information is insufficient, say so."
)

CHAT_FORMAT_VERSION = "chatml-v1"

SFT_DIR = Path("data/sft")
RAW_DIR = SFT_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("System instruction defined.")
print("Chat format:", CHAT_FORMAT_VERSION)

In [ ]:
import pandas as pd
import re
import hashlib

def as_text(value):
    """Convert a value to clean text, treating missing values as empty."""
    if value is None:
        return ""
    if not isinstance(value, (list, dict)):
        try:
            if pd.isna(value):
                return ""
        except (TypeError, ValueError):
            pass
    return value.strip() if isinstance(value, str) else ""

def get_sources(row):
    """Preserve the source field without inventing citations."""
    value = row.get("final_answer_sources", [])

    if isinstance(value, list):
        return value

    if isinstance(value, dict):
        return [value]

    if isinstance(value, str) and value.strip():
        try:
            parsed = json.loads(value)
            if isinstance(parsed, dict):
                return [parsed]
            if isinstance(parsed, list):
                return parsed
        except json.JSONDecodeError:
            pass

    return []

def make_ajrasakha_record(row, idx, split_name):
    question = as_text(row.get("question"))
    answer = as_text(row.get("final_answer"))

    if not question or not answer:
        raise ValueError(
            f"Empty question/answer in {split_name}, row {idx}"
        )

    record_id = f"ajrasakha_{split_name}_{idx:07d}"
    sources = get_sources(row)

    # Use a source URL as doc_id where possible, to support source-grouped splits.
    first_source = sources[0] if sources else {}
    source_url = (
        first_source.get("source", "")
        if isinstance(first_source, dict) else ""
    )
    doc_id = source_url.strip() if isinstance(source_url, str) and source_url.strip() else record_id

    metadata = {
        "task": "question_answering",
        "domain": "agriculture",
        "source": "ajrasakha-dataset-v1",
        "doc_id": doc_id,
        "final_answer_sources": sources,
    }

    # Preserve useful crop/location fields if the dataset contains them.
    for column in ["normalised_crop", "state", "district", "season"]:
        value = as_text(row.get(column))
        if value:
            metadata[column] = value

    return {
        "id": record_id,
        "messages": [
            {"role": "system", "content": DEFAULT_SYSTEM},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ],
        "metadata": metadata,
    }

In [ ]:
def convert_dataframe(df, split_name):
    records = []

    for idx, (_, row) in enumerate(df.iterrows()):
        records.append(
            make_ajrasakha_record(row, idx, split_name)
        )

    return records

train_records = convert_dataframe(train_clean, "train")
val_records = convert_dataframe(validation_clean, "val")
test_records = convert_dataframe(test_clean, "test")

def write_jsonl(path, records):
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

write_jsonl(RAW_DIR / "train.jsonl", train_records)
write_jsonl(RAW_DIR / "val.jsonl", val_records)
write_jsonl(RAW_DIR / "test.jsonl", test_records)

print("Train:", len(train_records))
print("Validation:", len(val_records))
print("Test:", len(test_records))
print("Saved to:", RAW_DIR.resolve())

In [ ]:
ALLOWED_ROLES = {"system", "user", "assistant"}

def validate_record(record):
    errors = []

    if not isinstance(record.get("id"), str) or not record["id"]:
        errors.append("missing id")

    messages = record.get("messages")
    if not isinstance(messages, list) or len(messages) < 2:
        return errors + ["messages missing or too short"]

    for message in messages:
        if not isinstance(message, dict):
            errors.append("message is not an object")
            continue

        if message.get("role") not in ALLOWED_ROLES:
            errors.append(f"invalid role: {message.get('role')}")

        if not isinstance(message.get("content"), str) or not message["content"].strip():
            errors.append("empty or invalid message content")

    body = [
        m for m in messages
        if isinstance(m, dict) and m.get("role") != "system"
    ]

    if not body or body[0].get("role") != "user":
        errors.append("first non-system message must be user")

    if body and body[-1].get("role") != "assistant":
        errors.append("last message must be assistant")

    for first, second in zip(body, body[1:]):
        if first.get("role") == second.get("role"):
            errors.append("consecutive non-system roles must alternate")
            break

    metadata = record.get("metadata", {})
    if metadata.get("task") not in {
        "question_answering", "explanation", "summarization",
        "extraction", "classification", "reasoning",
        "transformation", "multi_turn", "terminology", "procedure"
    }:
        errors.append("invalid task")

    if not metadata.get("source") or not metadata.get("doc_id"):
        errors.append("missing source or doc_id")

    return errors

all_records = train_records + val_records + test_records

invalid_records = [
    (record["id"], validate_record(record))
    for record in all_records
    if validate_record(record)
]

print("Total records:", len(all_records))
print("Invalid records:", len(invalid_records))

for record_id, errors in invalid_records[:10]:
    print(record_id, errors)

assert not invalid_records, "Fix invalid records before proceeding."

In [ ]:
CHAT_FORMAT_VERSION = "chatml-v1"

def format_conversation(messages, add_generation_prompt=False):
    text = ""

    for message in messages:
        role = message["role"]
        content = message["content"]
        text += f"<|im_start|>{role}\n{content}<|im_end|>\n"

    if add_generation_prompt:
        text += "<|im_start|>assistant\n"

    return text

In [ ]:
demo = [
    {"role": "user", "content": "What is attention?"},
    {"role": "assistant", "content": "Attention helps a model focus on relevant information."},
]

expected = (
    "<|im_start|>user\n"
    "What is attention?<|im_end|>\n"
    "<|im_start|>assistant\n"
    "Attention helps a model focus on relevant information.<|im_end|>\n"
)

assert format_conversation(demo) == expected

assert format_conversation(
    demo[:1], add_generation_prompt=True
).endswith("<|im_start|>assistant\n")

print("ChatML formatter tests passed.")

In [ ]:
sample = train_records[0]

print("Structured record:")
print(json.dumps(sample, indent=2, ensure_ascii=False))

print("\nFormatted conversation:")
print(format_conversation(sample["messages"]))

# Tokenization and assistant-only masking

In [ ]:
from pathlib import Path
from tokenizers import Tokenizer

TOKENIZER_PATH = Path("tokenizer/tokenizer_qwen40k.json")

assert TOKENIZER_PATH.is_file(), (
    f"Tokenizer file not found: {TOKENIZER_PATH.resolve()}"
)

tok = Tokenizer.from_file(str(TOKENIZER_PATH))

print("Tokenizer loaded:", TOKENIZER_PATH)
print("Vocabulary size:", tok.get_vocab_size())

In [ ]:
IM_START = tok.token_to_id("<|im_start|>")
IM_END = tok.token_to_id("<|im_end|>")

assert IM_START is not None, "Missing <|im_start|> in tokenizer."
assert IM_END is not None, "Missing <|im_end|> in tokenizer."

# The guide uses the end-of-message token for padding.
# Padding labels must always be masked.
PAD_ID = IM_END

print("IM_START:", IM_START)
print("IM_END:", IM_END)
print("PAD_ID:", PAD_ID)

In [ ]:
assert tok.get_vocab_size() <= QWEN3_CONFIG["vocab_size"], (
    f"Tokenizer size {tok.get_vocab_size()} exceeds model vocabulary "
    f"{QWEN3_CONFIG['vocab_size']}."
)

print("Tokenizer/model vocabulary check passed.")

In [ ]:
IGNORE = -100
SFT_SEQ_LEN = 2048

def enc(text):
    return tok.encode(
        text,
        add_special_tokens=False
    ).ids

def encode_message(message):
    role = message["role"]
    content = message["content"]

    # Message header: <|im_start|>role\n
    header_ids = [IM_START] + enc(role + "\n")

    # Message content plus its end marker.
    content_ids = enc(content) + [IM_END]

    # Newline following <|im_end|>.
    tail_ids = enc("\n")

    return header_ids, content_ids, tail_ids

In [ ]:
def tokenize_conversation(messages, max_len=SFT_SEQ_LEN):
    input_ids = []
    labels = []

    for message in messages:
        header_ids, content_ids, tail_ids = encode_message(message)

        is_assistant = message["role"] == "assistant"

        input_ids.extend(header_ids)
        input_ids.extend(content_ids)
        input_ids.extend(tail_ids)

        # Role header is never supervised.
        labels.extend([IGNORE] * len(header_ids))

        # Supervise assistant content and <|im_end|>.
        # Mask system and user content.
        if is_assistant:
            labels.extend(content_ids)
        else:
            labels.extend([IGNORE] * len(content_ids))

        # Trailing newline is not supervised.
        labels.extend([IGNORE] * len(tail_ids))

    if len(input_ids) > max_len:
        return None

    if len(input_ids) < 2:
        return None

    # Shift once for next-token prediction.
    input_ids = input_ids[:-1]
    labels = labels[1:]

    if len(input_ids) != len(labels):
        raise RuntimeError("Input and label lengths do not match.")

    if all(label == IGNORE for label in labels):
        return None

    return {
        "input_ids": input_ids,
        "labels": labels,
    }

In [ ]:
from collections import Counter

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

def build_split(path, max_len=SFT_SEQ_LEN):
    records = read_jsonl(path)

    items = []
    dropped_length = 0
    dropped_no_labels = 0

    for record in records:
        result = tokenize_conversation(
            record["messages"],
            max_len=max_len
        )

        if result is None:
            # Separate length failures from other failures for reporting.
            total_length = sum(
                len(encode_message(m)[0])
                + len(encode_message(m)[1])
                + len(encode_message(m)[2])
                for m in record["messages"]
            )

            if total_length > max_len:
                dropped_length += 1
            else:
                dropped_no_labels += 1
            continue

        items.append({
            "id": record["id"],
            "task": record["metadata"]["task"],
            "input_ids": result["input_ids"],
            "labels": result["labels"],
        })

    token_count = sum(len(x["input_ids"]) for x in items)
    supervised_count = sum(
        sum(label != IGNORE for label in x["labels"])
        for x in items
    )

    print(f"\n{path}")
    print("Input records:", len(records))
    print("Kept:", len(items))
    print("Dropped for length:", dropped_length)
    print("Dropped with no supervised labels:", dropped_no_labels)
    print("Mean sequence length:",
          round(token_count / max(1, len(items))))
    print("Supervised-token fraction:",
          f"{supervised_count / max(1, token_count):.1%}")

    return items

In [ ]:
train_items = build_split("data/sft/raw/train.jsonl")
val_items = build_split("data/sft/raw/val.jsonl")

# Keep the test split untouched during training and tuning.
# Tokenize it later when you are ready for final evaluation.

In [ ]:
def show_mask(item, max_tokens=500):
    visible = []

    for token_id, label in zip(
        item["input_ids"],
        item["labels"]
    ):
        if label == IGNORE:
            visible.append("·")
        else:
            visible.append(
                tok.decode([label], skip_special_tokens=False)
            )

    return "".join(visible)[:max_tokens]

assert train_items, "No training examples survived tokenization."

print(show_mask(train_items[0]))

In [ ]:
for item in train_items[:100]:
    assert len(item["input_ids"]) == len(item["labels"])
    assert len(item["input_ids"]) <= SFT_SEQ_LEN - 1
    assert any(label != IGNORE for label in item["labels"])

print("First 100 training examples passed the checks.")

In [ ]:
record = read_jsonl("data/sft/raw/train.jsonl")[0]

formatted_text = format_conversation(record["messages"])
encoded = tok.decode(
    enc(formatted_text),
    skip_special_tokens=False
)

print("Formatted conversation:")
print(formatted_text[:500])

print("\nTokenizer round-trip:")
print(encoded[:500])

In [ ]:
import torch

TOKENIZED_DIR = SFT_DIR / "tokenized"
TOKENIZED_DIR.mkdir(parents=True, exist_ok=True)

torch.save(train_items, TOKENIZED_DIR / "train.pt")
torch.save(val_items, TOKENIZED_DIR / "val.pt")

print("Saved tokenized train and validation splits.")

# Build the SFT Dataset and DataLoader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

PAD_ID = IM_END
IGNORE = -100

assert len(train_items) > 0, "Training split is empty."
assert len(val_items) > 0, "Validation split is empty."

print("Training examples:", len(train_items))
print("Validation examples:", len(val_items))
print("Padding token ID:", PAD_ID)

In [ ]:
class SFTDataset(Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        item = self.items[index]

        return (
            torch.tensor(item["input_ids"], dtype=torch.long),
            torch.tensor(item["labels"], dtype=torch.long),
        )

train_dataset = SFTDataset(train_items)
val_dataset = SFTDataset(val_items)

print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))

In [ ]:
def collate(batch):
    batch_size = len(batch)
    max_length = max(x.size(0) for x, _ in batch)

    input_ids = torch.full(
        (batch_size, max_length),
        PAD_ID,
        dtype=torch.long,
    )

    labels = torch.full(
        (batch_size, max_length),
        IGNORE,
        dtype=torch.long,
    )

    for i, (x, y) in enumerate(batch):
        input_ids[i, :len(x)] = x
        labels[i, :len(y)] = y

    return input_ids, labels

In [ ]:
MICRO_BS = 8

# num_workers=0 is a conservative starting point for notebooks.
# Increase later if data loading becomes a bottleneck.
NUM_WORKERS = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=MICRO_BS,
    shuffle=True,
    collate_fn=collate,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=MICRO_BS,
    shuffle=False,
    collate_fn=collate,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

print("Training batches per epoch:", len(train_loader))
print("Validation batches:", len(val_loader))

In [ ]:
x, y = next(iter(train_loader))

print("input_ids shape:", tuple(x.shape))
print("labels shape:   ", tuple(y.shape))

supervised_fraction = (y != IGNORE).float().mean().item()
padding_fraction = (x == PAD_ID).float().mean().item()

print(f"Supervised-token fraction: {supervised_fraction:.2%}")
print(f"PAD_ID fraction:           {padding_fraction:.2%}")

assert x.shape == y.shape
assert x.dtype == torch.long
assert y.dtype == torch.long

# Padded input positions must never be supervised.
pad_positions = x == PAD_ID
assert torch.all(y[pad_positions] == IGNORE)

# Ensure this batch contains at least one supervised token.
assert (y != IGNORE).any()

print("DataLoader sanity checks passed.")

In [ ]:
device = next(model.parameters()).device

x_device = x.to(device)

model.eval()
with torch.no_grad():
    logits = model(x_device)

print("Logits shape:", tuple(logits.shape))
print("Input shape: ", tuple(x_device.shape))

assert logits.shape[:2] == x_device.shape
assert logits.shape[-1] == QWEN3_CONFIG["vocab_size"]

print("Model forward-pass check passed.")

# Configure the SFT training run

In [ ]:
import random
import numpy as np
import torch

from dataclasses import dataclass, asdict
from pathlib import Path

@dataclass
class SFTConfig:
    seq_len: int = 2048
    micro_batch_size: int = 8
    grad_accum: int = 4
    epochs: int = 3

    lr: float = 1e-5
    min_lr_ratio: float = 0.10
    weight_decay: float = 0.01
    betas: tuple = (0.9, 0.95)

    warmup_ratio: float = 0.05
    max_grad_norm: float = 1.0

    eval_every: int = 100
    log_every: int = 10
    seed: int = 42

    amp_dtype: str = "bfloat16"

cfg = SFTConfig()

# Reproducibility
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

print(asdict(cfg))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

use_bf16 = (
    device.type == "cuda"
    and torch.cuda.is_bf16_supported()
)

if cfg.amp_dtype == "bfloat16" and not use_bf16:
    print("BF16 unavailable; use FP16 on CUDA or FP32 on CPU.")

In [ ]:
if device.type == "cuda" and use_bf16:
    amp_dtype = torch.bfloat16
elif device.type == "cuda":
    amp_dtype = torch.float16
else:
    amp_dtype = torch.float32

print("Selected compute dtype:", amp_dtype)

In [ ]:
# Assumes `model` is already loaded from your pretrained checkpoint.
# Do not replace it with a newly initialized model here.

decay_params = []
no_decay_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue

    name_lower = name.lower()

    if (
        name.endswith(".bias")
        or "norm" in name_lower
        or "ln_" in name_lower
    ):
        no_decay_params.append(param)
    else:
        decay_params.append(param)

optimizer = torch.optim.AdamW(
    [
        {"params": decay_params, "weight_decay": cfg.weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ],
    lr=cfg.lr,
    betas=cfg.betas,
)

print("Parameters with weight decay:", len(decay_params))
print("Parameters without weight decay:", len(no_decay_params))
print("Initial learning rate:", cfg.lr)

In [ ]:
import math

updates_per_epoch = len(train_loader) // cfg.grad_accum

if updates_per_epoch < 1:
    raise ValueError(
        "Not enough training batches for one complete "
        "gradient-accumulation update. Reduce grad_accum "
        "or increase the training data/batch count."
    )

total_steps = updates_per_epoch * cfg.epochs
warmup_steps = max(1, int(total_steps * cfg.warmup_ratio))

print("Batches per epoch:", len(train_loader))
print("Optimizer updates per epoch:", updates_per_epoch)
print("Total optimizer updates:", total_steps)
print("Warmup updates:", warmup_steps)

In [ ]:
from torch.optim.lr_scheduler import LambdaLR

def lr_multiplier(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps

    progress = (step - warmup_steps) / max(
        1, total_steps - warmup_steps
    )
    progress = min(max(progress, 0.0), 1.0)

    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))

    return cfg.min_lr_ratio + (
        1.0 - cfg.min_lr_ratio
    ) * cosine

scheduler = LambdaLR(optimizer, lr_lambda=lr_multiplier)

print("Scheduler initialized.")
print("Current learning rate:", optimizer.param_groups[0]["lr"])

In [ ]:
assert cfg.lr == 1e-5
assert cfg.grad_accum == 4
assert cfg.epochs == 3
assert cfg.seq_len == 2048

assert len(train_loader) > 0
assert len(val_loader) > 0

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Trainable parameters:", f"{trainable_params:,}")
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Effective batch size:",
      cfg.micro_batch_size * cfg.grad_accum)
print("Total optimizer steps:", total_steps)
print("Configuration checks passed.")

#Run supervised fine-tuning (SFT)

In [ ]:
import math
import torch
import torch.nn.functional as F

IGNORE = -100

@torch.no_grad()
def evaluate(model, loader, max_batches=None):
    model.eval()

    loss_sum = 0.0
    n_tokens = 0

    for batch_idx, (x, y) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        # Skip batches without assistant targets.
        supervised = (y != IGNORE).sum().item()
        if supervised == 0:
            continue

        if device.type == "cuda":
            with torch.autocast(
                device_type="cuda",
                dtype=amp_dtype
            ):
                logits = model(x)
        else:
            logits = model(x)

        if logits.shape[:2] != y.shape:
            raise ValueError(
                f"Logits shape {logits.shape} is incompatible "
                f"with labels shape {y.shape}"
            )

        batch_loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)).float(),
            y.reshape(-1),
            ignore_index=IGNORE,
            reduction="sum",
        )

        if not torch.isfinite(batch_loss):
            raise FloatingPointError(
                f"Non-finite validation loss at batch {batch_idx}"
            )

        loss_sum += batch_loss.item()
        n_tokens += supervised

    if n_tokens == 0:
        raise ValueError(
            "Validation produced zero supervised tokens. "
            "Inspect the label masks from Step 4."
        )

    val_loss = loss_sum / n_tokens

    return {
        "val_loss": val_loss,
        "val_ppl": math.exp(min(val_loss, 20)),
        "val_tokens": n_tokens,
    }

In [ ]:
import os
import json
import shutil
from dataclasses import asdict
from pathlib import Path

RUN_DIR = Path("checkpoints/sft_run_agriculture")
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Adjust these paths to match your actual project.
BASE_CHECKPOINT = "checkpoints/pretrained_domain_model.pt"
TOKENIZER_PATH = "tokenizer/tokenizer_qwen40k.json"

CHAT_FORMAT_VERSION = "chatml-v1"

def save_checkpoint(
    path,
    model,
    cfg,
    step,
    metrics,
    optimizer=None,
    scheduler=None,
    scaler=None,
):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    payload = {
        "model": model.state_dict(),
        "config": QWEN3_CONFIG,
        "sft_config": asdict(cfg),
        "chat_format_version": CHAT_FORMAT_VERSION,
        "base_checkpoint": BASE_CHECKPOINT,
        "tokenizer_path": TOKENIZER_PATH,
        "step": step,
        "metrics": metrics,
    }

    # Include training state only if you intend to resume training.
    if optimizer is not None:
        payload["optimizer"] = optimizer.state_dict()
    if scheduler is not None:
        payload["scheduler"] = scheduler.state_dict()
    if scaler is not None:
        payload["scaler"] = scaler.state_dict()

    tmp_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp_path)
    os.replace(tmp_path, path)

    print(f"Saved: {path}")

In [ ]:
import time
from contextlib import nullcontext

# Mixed-precision gradient scaler: enabled only for CUDA FP16.
scaler = torch.cuda.amp.GradScaler(
    enabled=(device.type == "cuda" and amp_dtype == torch.float16)
)

def sft_loss(logits, labels):
    return F.cross_entropy(
        logits.reshape(-1, logits.size(-1)).float(),
        labels.reshape(-1),
        ignore_index=IGNORE,
    )

# Evaluate the untouched SFT starting point before any updates.
baseline_metrics = evaluate(model, val_loader)
print("Step 0 validation:", baseline_metrics)

best_val = float("inf")
bad_evals = 0
global_step = 0
stop_training = False

history = []

model.train()
optimizer.zero_grad(set_to_none=True)

for epoch in range(cfg.epochs):
    print(f"\nEpoch {epoch + 1}/{cfg.epochs}")

    running_loss = 0.0
    running_batches = 0
    pending_micro_batches = 0

    for batch_idx, (x, y) in enumerate(train_loader):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        if (y != IGNORE).sum().item() == 0:
            continue

        # Use autocast on CUDA; run normal precision on CPU.
        amp_context = (
            torch.autocast(device_type="cuda", dtype=amp_dtype)
            if device.type == "cuda"
            else nullcontext()
        )

        with amp_context:
            logits = model(x)
            loss = sft_loss(logits, y)

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Non-finite training loss at epoch {epoch + 1}, "
                f"batch {batch_idx}"
            )

        # Accumulate gradients; optimizer updates happen later.
        scaler.scale(loss / cfg.grad_accum).backward()

        running_loss += loss.item()
        running_batches += 1
        pending_micro_batches += 1

        # Only update after a complete accumulation group.
        if pending_micro_batches < cfg.grad_accum:
            continue

        scaler.unscale_(optimizer)

        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            cfg.max_grad_norm,
        )

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        pending_micro_batches = 0
        global_step += 1

        if global_step % cfg.log_every == 0:
            train_loss = running_loss / max(1, running_batches)
            current_lr = optimizer.param_groups[0]["lr"]

            record = {
                "epoch": epoch + 1,
                "step": global_step,
                "train_loss": train_loss,
                "lr": current_lr,
                "grad_norm": float(grad_norm),
            }
            history.append(record)

            print(
                f"step={global_step}/{total_steps} "
                f"train_loss={train_loss:.4f} "
                f"lr={current_lr:.3e} "
                f"grad_norm={float(grad_norm):.3f}"
            )

            running_loss = 0.0
            running_batches = 0

        should_evaluate = (
            global_step % cfg.eval_every == 0
            or global_step == total_steps
        )

        if should_evaluate:
            metrics = evaluate(model, val_loader)
            metrics["epoch"] = epoch + 1
            metrics["step"] = global_step

            print(f"Validation at step {global_step}: {metrics}")

            save_checkpoint(
                RUN_DIR / "last.pt",
                model,
                cfg,
                global_step,
                metrics,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=scaler,
            )

            if metrics["val_loss"] < best_val:
                best_val = metrics["val_loss"]
                bad_evals = 0

                save_checkpoint(
                    RUN_DIR / "best.pt",
                    model,
                    cfg,
                    global_step,
                    metrics,
                )
            else:
                bad_evals += 1

            model.train()

            if bad_evals >= 2:
                print(
                    "Early stopping: validation loss did not improve "
                    "for two consecutive evaluations."
                )
                stop_training = True
                break

    if stop_training:
        break

# Always save the latest model, even if the last evaluation was
# not exactly at the final optimizer step.
final_metrics = evaluate(model, val_loader)

save_checkpoint(
    RUN_DIR / "last.pt",
    model,
    cfg,
    global_step,
    final_metrics,
    optimizer=optimizer,
    scheduler=scheduler,
    scaler=scaler,
)

# Save training logs and configuration alongside checkpoints.
with open(RUN_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)

with open(RUN_DIR / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(asdict(cfg), f, indent=2)

print("\nSFT run finished.")
print("Optimizer steps:", global_step)
print("Final validation:", final_metrics)
print("Checkpoint directory:", RUN_DIR)

In [ ]:
from pathlib import Path
import json
import torch

assert (RUN_DIR / "last.pt").exists(), "last.pt was not created"

if not (RUN_DIR / "best.pt").exists():
    print(
        "WARNING: best.pt was not created. Check whether any "
        "validation event occurred and whether training batches "
        "contained supervised tokens."
    )

with open(RUN_DIR / "metrics.json", encoding="utf-8") as f:
    history = json.load(f)

print("Logged training records:", len(history))
print("Last logged record:", history[-1] if history else "No records")
print("Artifacts:", [p.name for p in RUN_DIR.iterdir()])

# Reload the checkpoint and evaluate the fine-tuned model

In [ ]:
from pathlib import Path
import torch

BEST_CKPT = RUN_DIR / "best.pt"

if not BEST_CKPT.exists():
    raise FileNotFoundError(
        f"{BEST_CKPT} not found. Complete Step 7 successfully first."
    )

# Load checkpoint metadata and weights.
try:
    ckpt = torch.load(
        BEST_CKPT,
        map_location="cpu",
        weights_only=False,
    )
except TypeError:
    # Compatibility with older PyTorch versions.
    ckpt = torch.load(BEST_CKPT, map_location="cpu")

assert "model" in ckpt
assert "config" in ckpt
assert "sft_config" in ckpt
assert ckpt.get("chat_format_version") == "chatml-v1"

# Instantiate a fresh model using the saved architecture config.
model_sft = Qwen3Model(ckpt["config"])
missing, unexpected = model_sft.load_state_dict(
    ckpt["model"],
    strict=True,
)

model_sft = model_sft.to(device)
model_sft.eval()

print("Reload successful.")
print("Checkpoint step:", ckpt.get("step"))
print("Recorded metrics:", ckpt.get("metrics"))

In [ ]:
reloaded_metrics = evaluate(model_sft, val_loader)

print("Recorded checkpoint metrics:", ckpt["metrics"])
print("Metrics after reload:", reloaded_metrics)

recorded_loss = ckpt["metrics"]["val_loss"]
reloaded_loss = reloaded_metrics["val_loss"]

print("Absolute loss difference:", abs(recorded_loss - reloaded_loss))

assert abs(recorded_loss - reloaded_loss) < 1e-3, (
    "Reloaded validation loss differs from the recorded metric. "
    "Check the model, tokenizer, validation data, precision, "
    "and checkpoint metadata."
)

In [ ]:
BASE_CKPT = Path("checkpoints/pretrained_domain_model.pt")

if not BASE_CKPT.exists():
    raise FileNotFoundError(
        "Pretrained checkpoint not found. Set BASE_CKPT to "
        "your immutable pretrained checkpoint."
    )

try:
    base_ckpt = torch.load(
        BASE_CKPT, map_location="cpu", weights_only=False
    )
except TypeError:
    base_ckpt = torch.load(BASE_CKPT, map_location="cpu")

model_pre = Qwen3Model(base_ckpt["config"])
model_pre.load_state_dict(base_ckpt["model"], strict=True)
model_pre = model_pre.to(device).eval()

print("Pretrained baseline model loaded.")
print("SFT comparison model loaded:", model_sft is not None)

In [ ]:
from tokenizers import Tokenizer

TOKENIZER_PATH = Path("tokenizer/tokenizer_qwen40k.json")
tok = Tokenizer.from_file(str(TOKENIZER_PATH))

IM_END = tok.token_to_id("<|im_end|>")
assert IM_END is not None, "Tokenizer is missing <|im_end|>."

def encode_prompt(messages):
    prompt = format_conversation(
        messages,
        add_generation_prompt=True,
    )
    return tok.encode(prompt, add_special_tokens=False).ids

def generate_answer(model, messages, max_new_tokens=256):
    model.eval()
    ids = encode_prompt(messages)
    x = torch.tensor([ids], dtype=torch.long, device=device)

    # Adapt this key if your model config uses another name.
    context_len = ckpt["config"].get("context_length", 8192)

    generated = []
    stopped = False

    with torch.no_grad():
        for _ in range(max_new_tokens):
            x_in = x[:, -context_len:]

            if device.type == "cuda":
                with torch.autocast(
                    device_type="cuda", dtype=amp_dtype
                ):
                    logits = model(x_in)
            else:
                logits = model(x_in)

            next_id = logits[:, -1, :].float().argmax(
                dim=-1, keepdim=True
            )
            token_id = next_id.item()

            if token_id == IM_END:
                stopped = True
                break

            generated.append(token_id)
            x = torch.cat([x, next_id], dim=1)

    return tok.decode(generated), stopped

## prepare a manual_prompts.jsonl for smoke test
{"id":"ag_001","category":"normal_question","messages":[{"role":"user","content":"What causes late blight in potato?"}],"expected":"Explain the disease and its cause accurately."}
{"id":"ag_002","category":"ambiguous_question","messages":[{"role":"user","content":"How much pesticide should I apply?"}],"expected":"Ask for the product, crop, area and label details before giving a rate."}
{"id":"ag_003","category":"insufficient_information","messages":[{"role":"user","content":"Which fungicide should I use on my field?"}],"expected":"Ask for crop, symptoms, location and relevant conditions; do not invent a recommendation."}
{"id":"ag_004","category":"incorrect_premise","messages":[{"role":"user","content":"Why do wheat roots form nitrogen-fixing nodules?"}],"expected":"Correct the false premise accurately."}
{"id":"ag_005","category":"instruction_following","messages":[{"role":"user","content":"Explain integrated pest management in exactly 3 bullet points."}],"expected":"Return exactly three relevant bullet points."}

In [ ]:
import json
import pandas as pd

BENCHMARK_PATH = Path("data/eval/manual_prompts.jsonl")
REPORT_DIR = Path("reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

with open(BENCHMARK_PATH, encoding="utf-8") as f:
    benchmark = [json.loads(line) for line in f if line.strip()]

def run_generation_benchmark(model, tag):
    rows = []

    for item in benchmark:
        prediction, stopped = generate_answer(
            model,
            item["messages"],
            max_new_tokens=256,
        )

        rows.append({
            "id": item["id"],
            "category": item["category"],
            "prediction": prediction,
            "expected_behavior": item["expected"],
            "stopped": stopped,
        })

    df = pd.DataFrame(rows)
    df.to_csv(REPORT_DIR / f"generations_{tag}.csv", index=False)
    return df

df_before = run_generation_benchmark(model_pre, "pretrained")
df_after = run_generation_benchmark(model_sft, "sft")

print("PRETRAINED MODEL")
display(df_before)

print("SFT MODEL")
display(df_after)